Imports - Cell responsible for libraries that the notebook needs. 

In [ ]:
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import IPython.display as ipd
import numpy as np
import torch
import json
import os
import noisereduce as nr

from hparams import create_hparams
from model import Tacotron2
from text import text_to_sequence

print("Imports complete")

✓ Imports complete


Load Tacotron2 - Cell responsible for getting the trained model. It then points to and loads the checkpoint file, after which the weights are added to the model. It also detects if a GPU is accessible and finally puts the model in inference mode.

In [ ]:
print("Loading Tacotron2")
hparams = create_hparams()
checkpoint_path = "checkpoints_SP1/checkpoint_258000"

tacotron2 = Tacotron2(hparams)
checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')
tacotron2.load_state_dict(checkpoint_dict['state_dict'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tacotron2 = tacotron2.to(device)
tacotron2.eval()

print(f"Tacotron2 loaded from iteration {checkpoint_dict.get('iteration', 'unknown')}")
print(f"Device: {device}")

Loading Tacotron2...
✓ Tacotron2 loaded from iteration 258000
✓ Device: cuda


Load Hifi-gan - Cell responsible for the Hifi-gan vocoder. Due to Hifi-gan and Tacotron2 having some same named files it clears those before adding the Hifi-gan versions to the system path. It then loads the checkpoint and does the GPU detection after which it enables inference mode. Finally, it removes the HIfi-gan directory from the system path. 

In [ ]:
print("Loading HiFi-GAN")

# Paths
hifigan_dir = os.path.abspath("C:/Users/riyad/Desktop/3020/TTS_fyp/hifi-gan_SP")
hifigan_checkpoint = os.path.join(hifigan_dir, "cp_hifigan_SP/g_00930000")
hifigan_config = os.path.join(hifigan_dir, "config_SP.json")

# Load config
with open(hifigan_config) as f:
    config = json.load(f)

# Create config object
class HiFiGANConfig:
    def __init__(self, config_dict):
        for key, value in config_dict.items():
            setattr(self, key, value)

h = HiFiGANConfig(config)

# CRITICAL FIX: Clear any cached imports of 'utils' and 'models'
import sys
for module_name in list(sys.modules.keys()):
    if module_name in ['utils', 'models', 'env']:
        del sys.modules[module_name]

# Add HiFi-GAN to path
sys.path.insert(0, hifigan_dir)

try:
    # Now import fresh (will find HiFi-GAN's utils.py)
    from models import Generator
    
    # Load model
    hifigan = Generator(h)
    state_dict = torch.load(hifigan_checkpoint, map_location='cpu')
    hifigan.load_state_dict(state_dict['generator'])
    hifigan = hifigan.to(device)
    hifigan.eval()
    hifigan.remove_weight_norm()
    
    print("HiFi-GAN loaded successfully")
    
finally:
    # Remove from sys.path
    sys.path.remove(hifigan_dir)

print(f"Using device: {device}")

Loading HiFi-GAN...
Removing weight norm...
✓ HiFi-GAN loaded successfully
✓ Using device: cuda


C:\Users\riyad\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Synthesis - Cell responsible for taking the text and converting it to audio. 

In [ ]:
def synthesize(text):
    """Generate speech from text with visualizations"""
    
    print(f"Synthesizing: '{text}'")
    
    # Text to sequence
    sequence = np.array(text_to_sequence(text, ['english_cleaners']))[None, :]
    sequence = torch.from_numpy(sequence).to(device).long()
    
    # Generate mel with Tacotron2
    with torch.no_grad():
        mel_outputs, mel_outputs_postnet, _, alignments = tacotron2.inference(sequence)
    
    print(f"  Mel shape: {mel_outputs_postnet.shape}")
    
    # Plot alignment
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # Alignment plot
    im1 = ax1.imshow(alignments[0].cpu().numpy().T, 
                     aspect='auto', origin='lower', 
                     interpolation='none', cmap='viridis')
    ax1.set_xlabel('Decoder timestep')
    ax1.set_ylabel('Encoder timestep')
    ax1.set_title('Attention Alignment (should show diagonal line)')
    plt.colorbar(im1, ax=ax1)
    
    # Mel-spectrogram plot
    im2 = ax2.imshow(mel_outputs_postnet[0].cpu().numpy(), 
                     aspect='auto', origin='lower', 
                     interpolation='none', cmap='viridis')
    ax2.set_xlabel('Time frames')
    ax2.set_ylabel('Mel channels')
    ax2.set_title('Generated Mel-Spectrogram')
    plt.colorbar(im2, ax=ax2)
    
    plt.tight_layout()
    plt.show()
    
    # Generate audio with HiFi-GAN
    print("Converting mel to audio")
    with torch.no_grad():
        audio = hifigan(mel_outputs_postnet).squeeze()
    
    audio = audio.cpu().numpy()
    
    print("Applying noise reduction")
    audio = nr.reduce_noise(y=audio, sr=22050, stationary=True, prop_decrease=0.9)
    print("Noise reduction applied")

    # Normalize
    audio = audio / np.max(np.abs(audio))
    
    print(f"  Audio length: {len(audio) / 22050:.2f} seconds")
    print("Done!")
    
    return audio

print("Synthesis function defined")

✓ Synthesis function defined


Test - Cell responsible for getting output. You enter a name in the text space then it is passed through the synthesize function. A player is made in the notebook at the same sample rate as Tacotron2 and Hifi-gan to hear the generated audio. 

In [ ]:
text = "William Perkins."
audio = synthesize(text)

# Play audio
ipd.Audio(audio, rate=22050)

Synthesizing: 'Arthur Wint.'
  Mel shape: torch.Size([1, 80, 140])
  Converting mel to audio...
  Applying noise reduction...
  ✓ Noise reduction applied
  Audio length: 1.63 seconds
✓ Done!


C:\Users\riyad\AppData\Local\Temp\ipykernel_17196\1021986336.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
